# KRUMP_CORE_V1 — Pilot 20 Epoch\n\nQuick LoRAの成功成果物を復元し、Corrected LoRAで最大20 epochのみを実行します。50 epoch・Benchmarkは含めません。

In [ ]:
# 1. Ephemeral Colab-only paths: no persistent cloud storage
from pathlib import Path
import json, os, re, shutil, subprocess, sys, time, traceback, zipfile
WORK=Path('/content/KRUMP_CORE_V1_OUTPUT')
RUNTIME=Path('/content/KRUMP_CORE_V1_RUNTIME')
INPUT_ROOT=Path('/content/KRUMP_DATASET_V1')
WORK.mkdir(parents=True,exist_ok=True); RUNTIME.mkdir(parents=True,exist_ok=True); INPUT_ROOT.mkdir(parents=True,exist_ok=True)
os.environ.update({'UV_CACHE_DIR':str(RUNTIME/'uv-cache'),'PIP_NO_CACHE_DIR':'1','HF_HOME':str(RUNTIME/'hf-cache'),'HUGGINGFACE_HUB_CACHE':str(RUNTIME/'hf-cache'/'hub'),'MPLBACKEND':'Agg'})
LOG=WORK/'logs'/'smoke.log'; LOG.parent.mkdir(exist_ok=True)
def run(name,args,cwd=None,timeout=7200,input_text=None):
  p=subprocess.run(args,cwd=cwd,input=input_text,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout,check=False)
  LOG.open('a',encoding='utf-8').write(f'\n===== {name} =====\n{p.stdout}\n'); print(p.stdout[-6000:])
  if p.returncode: raise RuntimeError(f'{name} failed (exit={p.returncode}); see {LOG}')
  return p.stdout
def fail(stage,e):
  (WORK/'FAILED.json').write_text(json.dumps({'stage':stage,'error':str(e),'traceback':traceback.format_exc()},indent=2),encoding='utf-8'); raise e


In [ ]:
# 2. CUDA fail-fast: only T4 / L4 / A100 may train
try:
 import torch
 print(subprocess.check_output(['nvidia-smi','-L'],text=True)); print('torch=',torch.__version__)
 if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable')
 GPU=torch.cuda.get_device_name(0); print('GPU=',GPU); print('arch=',torch.cuda.get_arch_list())
 if not any(x in GPU for x in ('T4','L4','A100')): raise RuntimeError(f'Unsupported GPU: {GPU}; training not started')
except Exception as e: fail('gpu_preflight',e)


In [ ]:
# 3. Restore the downloaded Smoke ZIP only when the Quick artifacts are not already present\ntry:\n from google.colab import files\n QUICK=WORK/'quick_lora'; TENSORS=WORK/'preprocess'/'tensors'/'KRUMP_DATASET_V1'\n if not ((QUICK/'final').exists() and len(list(TENSORS.rglob('*.pt'))) >= 18):\n  print('Upload only KRUMP_CORE_V1_SMOKE_OUTPUT.zip from the completed Smoke run.')\n  uploaded=files.upload()\n  if len(uploaded) != 1: raise RuntimeError('Upload exactly one Smoke output ZIP.')\n  name,data=next(iter(uploaded.items()))\n  if not name.lower().endswith('.zip'): raise RuntimeError('Expected a ZIP archive.')\n  incoming=Path('/content')/name; incoming.write_bytes(data)\n  shutil.unpack_archive(str(incoming),str(WORK))\n if len(list(TENSORS.rglob('*.pt'))) < 18: raise RuntimeError('Smoke tensors missing or incomplete.')\n if not (QUICK/'final').exists() and not list((QUICK/'checkpoints').rglob('*')): raise RuntimeError('Quick adapter/checkpoint missing.')\n print('SMOKE_RESTORE_PASS',{'tensors':len(list(TENSORS.rglob('*.pt'))),'quick':str(QUICK)})\nexcept Exception as e: fail('restore_smoke',e)\n

In [ ]:
# 4. Same ACE-Step 1.5 runtime and base model as Smoke Pipeline
try:
 ACE=RUNTIME/'ACE-Step-1.5'; CKPT=RUNTIME/'checkpoints'
 run('install uv',['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],timeout=300)
 UV=shutil.which('uv') or '/root/.local/bin/uv'
 if not ACE.exists(): run('clone',['git','clone','https://github.com/ace-step/ACE-Step-1.5.git',str(ACE)],timeout=900)
 run('pin',['git','checkout','7202bc354d7fc31d1c0e5a90b0b49fb610e52362'],cwd=ACE,timeout=120)
 run('deps',[UV,'sync','--no-cache'],cwd=ACE,timeout=2400); CKPT.mkdir(exist_ok=True)
 link=ACE/'checkpoints'
 if link.exists() or link.is_symlink(): link.unlink() if (link.is_symlink() or link.is_file()) else shutil.rmtree(link)
 link.symlink_to(CKPT,target_is_directory=True)
 run('base model',[UV,'run','--no-sync','acestep-download','--model','acestep-v15-base','--dir',str(CKPT)],cwd=ACE,timeout=3600)
 if not any(CKPT.rglob('*.safetensors')): raise RuntimeError('Base weights missing')
 run('import',[UV,'run','python','-c','import acestep;print("ACE_STEP_IMPORT_PASS")'],cwd=ACE,timeout=300)
except Exception as e: fail('setup',e)


In [ ]:
# 5. Corrected LoRA Pilot: rank64 / alpha128 / batch1 / checkpoint epochs 5, 10, 15, 20\ntry:\n PILOT=WORK/'pilot_20'; CKS=PILOT/'checkpoints'; FINAL=PILOT/'final'; PILOT.mkdir(parents=True,exist_ok=True)\n def latest_checkpoint(root):\n  dirs=[p for p in root.glob('epoch_*') if p.is_dir()] if root.exists() else []\n  def key(p):\n   m=re.search(r'epoch_(\\d+)',p.name); return int(m.group(1)) if m else -1\n  return max(dirs,key=key) if dirs else None\n if not FINAL.exists():\n  SAFE_TENSORS=ACE/'runtime_dataset'/'KRUMP_DATASET_V1'\n  if not SAFE_TENSORS.exists(): shutil.copytree(TENSORS,SAFE_TENSORS)\n  resume_from=latest_checkpoint(CKS) or latest_checkpoint(QUICK/'checkpoints')\n  if resume_from is None: raise RuntimeError('No Quick/Pilot checkpoint available for official resume.')\n  config={'training':'Corrected LoRA','rank':64,'alpha':128,'batch_size':1,'gradient_accumulation':4,'epochs':20,'save_every':5,'resume_from':str(resume_from),'seed':42}\n  (PILOT/'pilot_20_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')\n  torch.cuda.reset_peak_memory_stats()\n  cmd=[UV,'run','python','train.py','fixed','--checkpoint-dir',str(CKPT),'--model-variant','base','--dataset-dir',str(SAFE_TENSORS),'--output-dir',str(PILOT),'--adapter-type','lora','--rank','64','--alpha','128','--dropout','0.15','--batch-size','1','--gradient-accumulation','4','--epochs','20','--save-every','5','--lr','5e-5','--seed','42','--log-every','1','--device','cuda:0','--precision','bf16','--gradient-checkpointing','--offload-encoder','--resume-from',str(resume_from)]\n  run('pilot corrected lora',cmd,cwd=ACE,timeout=39600,input_text='y\\n')\n text=LOG.read_text(encoding='utf-8',errors='replace'); losses=re.findall(r'(?i)loss[^0-9]*([0-9]+(?:\\.[0-9]+)?)',text)\n checkpoints=[str(x) for x in CKS.glob('epoch_*') if x.is_dir()]\n adapters=[str(x) for x in FINAL.rglob('*')] if FINAL.exists() else []\n report={'status':'PASS' if losses and checkpoints and adapters else 'FAIL','losses':losses[-50:],'checkpoint_paths':checkpoints,'adapter_path':str(FINAL),'resume':True,'peak_vram_gib':round(torch.cuda.max_memory_allocated()/1024**3,3)}\n (PILOT/'training_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')\n if report['status'] != 'PASS': raise RuntimeError('Pilot missing numeric loss, checkpoint, or final adapter.')\n stale=PILOT/'FAILED.json'\n if stale.exists(): stale.unlink()\n print(json.dumps(report,indent=2))\nelse:\n report=json.loads((PILOT/'training_report.json').read_text(encoding='utf-8')) if (PILOT/'training_report.json').exists() else {'status':'PASS','adapter_path':str(FINAL),'resume':True}\n print('PILOT_ALREADY_COMPLETE',json.dumps(report,indent=2))\nexcept Exception as e: fail('pilot_20',e)\n

In [ ]:
# 6. Package Pilot artifacts for download before the Colab session ends\nfrom google.colab import files\narchive=Path('/content/KRUMP_CORE_V1_PILOT_20_OUTPUT.zip')\nif archive.exists(): archive.unlink()\nshutil.make_archive(str(archive.with_suffix('')),'zip',WORK)\nif not archive.exists() or archive.stat().st_size == 0: raise RuntimeError('Pilot output ZIP was not created.')\nprint({'zip':str(archive),'bytes':archive.stat().st_size})\nfiles.download(str(archive))\n